# Budget and Expense Analysis
**Important:** Included CSV files are synthetic demonstration data created to test the workflow. They are not real company records. Replace them with verified datasets before making business claims.

## Objectives
- Clean budget and expense records.
- Merge on month, department, and expense category.
- Calculate budget variance and utilization.
- Analyze monthly, department, and category spending.
- Flag over-budget combinations and export results.

In [ ]:
# Budget & Expense Analysis
# IMPORTANT: The included CSVs are synthetic demonstration data, not real company records.

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / "data"
budget = pd.read_csv(DATA / "budget.csv")
expenses = pd.read_csv(DATA / "expenses.csv")

# 1. Inspect and clean
print("Budget shape:", budget.shape)
print("Expenses shape:", expenses.shape)
display(budget.head())
display(expenses.head())

for df in (budget, expenses):
    df["Month"] = pd.to_datetime(df["Month"], errors="coerce")
    df["Department"] = df["Department"].astype("string").str.strip()
    df["Expense_Category"] = df["Expense_Category"].astype("string").str.strip()

budget["Approved_Budget"] = pd.to_numeric(budget["Approved_Budget"], errors="coerce")
expenses["Actual_Expense"] = pd.to_numeric(expenses["Actual_Expense"], errors="coerce")
budget = budget.dropna(subset=["Month", "Department", "Expense_Category", "Approved_Budget"])
expenses = expenses.dropna(subset=["Month", "Department", "Expense_Category", "Actual_Expense"])

# 2. Aggregate duplicate keys, then merge using shared keys.
keys = ["Month", "Department", "Expense_Category"]
budget_g = budget.groupby(keys, as_index=False)["Approved_Budget"].sum()
expenses_g = expenses.groupby(keys, as_index=False)["Actual_Expense"].sum()
analysis = budget_g.merge(expenses_g, on=keys, how="outer", validate="one_to_one")
analysis[["Approved_Budget", "Actual_Expense"]] = analysis[["Approved_Budget", "Actual_Expense"]].fillna(0)

# 3. Financial KPIs
analysis["Variance"] = analysis["Approved_Budget"] - analysis["Actual_Expense"]
analysis["Budget_Utilization_Pct"] = analysis.apply(
    lambda row: row["Actual_Expense"] / row["Approved_Budget"] * 100
    if row["Approved_Budget"] > 0 else pd.NA, axis=1)
analysis["Over_Budget"] = analysis["Actual_Expense"] > analysis["Approved_Budget"]

total_budget = analysis["Approved_Budget"].sum()
total_actual = analysis["Actual_Expense"].sum()
overall_variance = total_budget - total_actual
overall_utilization = (total_actual / total_budget * 100) if total_budget else 0

print(f"Total approved budget: PKR {total_budget:,.2f}")
print(f"Total actual expenses: PKR {total_actual:,.2f}")
print(f"Budget variance (budget - actual): PKR {overall_variance:,.2f}")
print(f"Budget utilization: {overall_utilization:.2f}%")
print(f"Over-budget month/category/department combinations: {int(analysis['Over_Budget'].sum())}")

# 4. Department and category analysis
department_summary = analysis.groupby("Department", as_index=False)[
    ["Approved_Budget", "Actual_Expense", "Variance"]].sum()
department_summary["Utilization_Pct"] = (
    department_summary["Actual_Expense"] / department_summary["Approved_Budget"] * 100
)
display(department_summary.sort_values("Variance"))

category_summary = analysis.groupby("Expense_Category", as_index=False)[
    ["Approved_Budget", "Actual_Expense", "Variance"]].sum()
category_summary["Utilization_Pct"] = (
    category_summary["Actual_Expense"] / category_summary["Approved_Budget"] * 100
)
display(category_summary.sort_values("Actual_Expense", ascending=False))

# 5. Monthly trend
monthly_summary = analysis.groupby("Month", as_index=False)[
    ["Approved_Budget", "Actual_Expense", "Variance"]].sum()
display(monthly_summary)

# 6. Visualizations
monthly_summary.plot(x="Month", y=["Approved_Budget", "Actual_Expense"], marker="o",
                     title="Approved Budget vs Actual Expenses", figsize=(10, 5))
plt.ylabel("PKR")
plt.tight_layout()
plt.show()

department_summary.set_index("Department")[["Approved_Budget", "Actual_Expense"]].plot(
    kind="bar", title="Department-wise Spending", figsize=(10, 5))
plt.ylabel("PKR")
plt.tight_layout()
plt.show()

category_summary.set_index("Expense_Category")["Actual_Expense"].sort_values().plot(
    kind="barh", title="Actual Expense by Category", figsize=(9, 5))
plt.xlabel("PKR")
plt.tight_layout()
plt.show()

monthly_summary.plot(x="Month", y="Variance", kind="bar",
                     title="Monthly Budget Variance", figsize=(10, 5))
plt.ylabel("PKR (positive = under budget)")
plt.tight_layout()
plt.show()

# 7. Over-budget alerts and exportable result
alerts = analysis.loc[analysis["Over_Budget"]].sort_values("Variance")
display(alerts)
analysis.to_csv(ROOT / "analysis_output.csv", index=False)
department_summary.to_csv(ROOT / "department_summary.csv", index=False)
category_summary.to_csv(ROOT / "category_summary.csv", index=False)

# 8. Interpretation guardrail
print("DATA LIMITATION: Any patterns above describe synthetic demonstration data only.")
print("Do not present these results as findings about a real company. Replace the CSVs with verified records, rerun, and rewrite the report.")
